# Simulating the ABQLBM with a `YMonomial` object

`YMonomial` describes a solid region by the polynomial predicate $y \lessgtr x^k$ rather than by a
cuboid, which the algorithm resolves with an arithmetic comparison oracle instead of a range check.
Curved boundaries are therefore expressible without approximating them by boxes. Only $k = 2$ is
implemented so far, and the grid must satisfy $n_x \leq n_y$ (square or tall).

`qlbm` builds its circuits as `qarp` blocks and simulates them with `qarp`'s statevector simulator.
A run is described by a `SimulationConfig`, which ties the four algorithmic circuits — initial
conditions, time step algorithm, postprocessing, and measurement — to a compiler optimization level
and a shot budget. `cfg.validate()` type-checks the configuration and `cfg.prepare_for_simulation()`
lowers every component into a built `qarp` block. A `QarpRunner` then drives the time loop and writes
one ParaView frame per time step into `output_dir/paraview`.

`statevector_snapshots=True` is the fast path: the runner hands the state produced at step $k$ back to
the simulator as the *initial state* of step $k+1$, so simulating $n$ steps costs $n$ applications of
the algorithm instead of $\frac{n(n+1)}{2}$. Passing `qarp.EXACT` instead of a shot count returns exact
outcome probabilities rather than sampled counts.

> **Demo-sized.** The monomial oracle needs a register wide enough to hold $y^k$, so its
> gate count grows quickly with the grid: the 4x16 domain below runs in seconds, while 8x16
> already costs tens of seconds per time step. Simulation cost grows exponentially in the number of qubits, so scale the lattice and
> `NUM_STEPS` up deliberately.

In [ ]:
from qlbm.components import (
    EmptyPrimitive,
)
from qlbm.components.ab.ab import ABQLBM
from qlbm.components.ab.initial import ABDiscreteUniformInitialConditions
from qlbm.components.ab.measurement import ABGridMeasurement
from qlbm.infra import QarpRunner, SimulationConfig
from qlbm.lattice import ABLattice
from qlbm.tools.utils import create_directory_and_parents

lattice = ABLattice(
    {
        "lattice": {"dim": {"x": 4, "y": 16}, "velocities": "D2Q9"},
        "geometry": [
            {
                "shape": "ymonomial",
                "exponent": 2,
                "comparator": "<",
                "boundary": "bounceback",
            }
        ],
    }
)

output_dir = "qlbm-output/ab-d2q9-4x16-1-obstacle-ymonomial-qarp"
create_directory_and_parents(output_dir)

In [ ]:
# Number of shots to sample for each time step
NUM_SHOTS = 2**12

# Number of time steps to simulate
NUM_STEPS = 8

In [ ]:
cfg = SimulationConfig(
    initial_conditions=ABDiscreteUniformInitialConditions(
        lattice,
        [1, 5, 8],
        ([], list(range(lattice.num_gridpoints[1].bit_length()))),
    ),
    algorithm=ABQLBM(lattice, True),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABGridMeasurement(lattice),
    optimization_level=0,
    shots=NUM_SHOTS,
)

cfg.validate()
cfg.prepare_for_simulation()

In [ ]:
runner = QarpRunner(cfg, lattice, seed=1234)


# Snapshots carry the statevector between time steps, making the loop O(NUM_STEPS)
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)